In [1]:
import numpy as np
import pandas as pd
import os
import tokenizers
import string
import torch
import transformers
import torch.nn as nn
from torch.nn import functional as F
from tqdm import tqdm
import re
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from tokenizers import ByteLevelBPETokenizer
from transformers import AutoTokenizer, AutoConfig, AutoModel, PreTrainedModel
from torch.optim.swa_utils import AveragedModel, SWALR
from torch.utils.data import Sampler

seed everything we have

In [2]:
MAX_LEN = 192
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 8
MODEL_NAME = "/kaggle/input/models/cbdsigmas/deberta-v3-large/pytorch/default/1/deberta-v3-large"
TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

MODEL_NAME2 = "/kaggle/input/datasets/abhishek/roberta-base"
TOKENIZER2 = ByteLevelBPETokenizer(
    f"{MODEL_NAME2}/vocab.json",
    f"{MODEL_NAME2}/merges.txt",
    lowercase=True,
    add_prefix_space=True
)
BASE_SEED = 42
EPOCHS = 4

In [3]:
def seed_everything(seed):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DeBERTaV3-large: model 1

In [4]:
class TweetModel(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.roberta = transformers.AutoModel.from_pretrained(
            MODEL_NAME,
            config=conf,
            attn_implementation="eager",
            torch_dtype=torch.float32
        )

        hidden = conf.hidden_size

        self.layer_norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 2)

        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, ids, mask, token_type_ids=None):
        # DeBERTa-v3 does not use token_type_ids; avoid passing them.
        outputs = self.roberta(
            input_ids=ids,
            attention_mask=mask,
            return_dict=False
        )

        sequence_output = outputs[0]

        # Safety: avoid propagating non-finite backbone activations.
        if not torch.isfinite(sequence_output).all():
            sequence_output = torch.nan_to_num(sequence_output, nan=0.0, posinf=1e4, neginf=-1e4)

        with torch.cuda.amp.autocast(enabled=False):
            x = self.layer_norm(sequence_output.float())
            x = self.dropout(x)
            logits = self.classifier(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)

        start_logits, end_logits = logits.split(1, dim=-1)
        return start_logits.squeeze(-1), end_logits.squeeze(-1)


RoBERTa-base: model 2

In [5]:
class TweetModel2(transformers.BertPreTrainedModel):
    def __init__(self, conf):
        super().__init__(conf)
        self.roberta = transformers.RobertaModel.from_pretrained(
            MODEL_NAME2,
            config=conf
        )
        self.drop_out = nn.Dropout(0.1)
        self.l0 = nn.Linear(768 * 2, 2)
        torch.nn.init.normal_(self.l0.weight, std=0.02)

    def forward(self, ids, mask, token_type_ids=None):
        outputs = self.roberta(
            ids,
            attention_mask=mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True
        )

        hs = outputs.hidden_states
        out = torch.cat((hs[-1], hs[-2]), dim=-1)

        out = self.drop_out(out)
        logits = self.l0(out)

        start_logits, end_logits = logits.split(1, dim=-1)
        return start_logits.squeeze(-1), end_logits.squeeze(-1)

post-preprocessing (copied, not magic)

In [6]:
def pp(filtered_output, real_tweet):
    filtered_output = ' '.join(filtered_output.split())
    if len(real_tweet.split()) < 2:
        filtered_output = real_tweet
    else:
        if len(filtered_output.split()) == 1:
            if filtered_output.endswith(".."):
                if real_tweet.startswith(" "):
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\.)\1{2,}', '', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\.)\1{2,}', '.', filtered_output)
                else:
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\.)\1{2,}', '.', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\.)\1{2,}', '..', filtered_output)
                return filtered_output
            if filtered_output.endswith('!!'):
                if real_tweet.startswith(" "):
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\!)\1{2,}', '', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\!)\1{2,}', '!', filtered_output)
                else:
                    st = real_tweet.find(filtered_output)
                    fl = real_tweet.find("  ")
                    if fl != -1 and fl < st:
                        filtered_output = re.sub(r'(\!)\1{2,}', '!', filtered_output)
                    else:
                        filtered_output = re.sub(r'(\!)\1{2,}', '!!', filtered_output)
                return filtered_output

        if real_tweet.startswith(" "):
            filtered_output = filtered_output.strip()
            text_annotetor = ' '.join(real_tweet.split())
            start = text_annotetor.find(filtered_output)
            end = start + len(filtered_output)
            start -= 0
            end += 2
            flag = real_tweet.find("  ")
            if flag < start:
                filtered_output = real_tweet[start:end]

        if "  " in real_tweet and not real_tweet.startswith(" "):
            filtered_output = filtered_output.strip()
            text_annotetor = re.sub(" {2,}", " ", real_tweet)
            start = text_annotetor.find(filtered_output)
            end = start + len(filtered_output)
            start -= 0
            end += 2
            flag = real_tweet.find("  ")
            if flag < start:
                filtered_output = real_tweet[start:end]
    return filtered_output

In [7]:
def process_data(tweet, selected_text, sentiment, tokenizer, max_len):
    tweet = " " + " ".join(str(tweet).split())
    selected_text = " " + " ".join(str(selected_text).split())

    # locate selected text
    len_st = len(selected_text) - 1
    idx0, idx1 = None, None

    for ind in (i for i, e in enumerate(tweet) if e == selected_text[1]):
        if " " + tweet[ind: ind+len_st] == selected_text:
            idx0 = ind
            idx1 = ind + len_st - 1
            break

    char_targets = [0] * len(tweet)
    if idx0 is not None and idx1 is not None:
        for ct in range(idx0, idx1 + 1):
            char_targets[ct] = 1

    # reserve 5 tokens: [CLS] sentiment [SEP] [SEP] ... [SEP]
    max_tweet_tokens = max_len - 5

    # encode tweet WITH truncation to keep shapes valid
    tweet_enc = tokenizer(
        tweet,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_tweet_tokens
    )

    input_ids_orig = tweet_enc["input_ids"]
    tweet_offsets = tweet_enc["offset_mapping"]

    # encode sentiment token
    sentiment_enc = tokenizer(
        sentiment,
        add_special_tokens=False
    )
    sentiment_id = sentiment_enc["input_ids"][0]

    # build final sequence (same as Kaggle)
    input_ids = (
        [tokenizer.cls_token_id] +
        [sentiment_id] +
        [tokenizer.sep_token_id] +
        [tokenizer.sep_token_id] +
        input_ids_orig +
        [tokenizer.sep_token_id]
    )

    offsets = (
        [(0,0)] * 4 +
        tweet_offsets +
        [(0,0)]
    )

    # find target indices
    target_idx = []
    for j, (o1, o2) in enumerate(tweet_offsets):
        if sum(char_targets[o1:o2]) > 0:
            target_idx.append(j)

    if len(target_idx) > 0:
        targets_start = target_idx[0] + 4
        targets_end = target_idx[-1] + 4
    else:
        targets_start = 0
        targets_end = 0

    # clamp in case selected span was truncated
    max_idx = len(input_ids) - 1
    targets_start = min(targets_start, max_idx)
    targets_end = min(targets_end, max_idx)

    # defensive truncate (should already be <= max_len)
    input_ids = input_ids[:max_len]
    offsets = offsets[:max_len]

    # padding
    padding_length = max_len - len(input_ids)
    mask = [1] * len(input_ids) + [0] * padding_length
    input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
    token_type_ids = [0] * max_len
    offsets = offsets + [(0,0)] * padding_length

    return {
        "ids": input_ids,
        "mask": mask,
        "token_type_ids": token_type_ids,
        "targets_start": targets_start,
        "targets_end": targets_end,
        "orig_tweet": tweet,
        "orig_selected": selected_text,
        "sentiment": sentiment,
        "offsets": offsets
    }

class TweetDataset:
    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.tokenizer = TOKENIZER
        self.max_len = MAX_LEN
    
    def __len__(self):
        return len(self.tweet)

    def __getitem__(self, item):
        data = process_data(
            self.tweet[item], 
            self.selected_text[item], 
            self.sentiment[item],
            self.tokenizer,
            self.max_len
        )

        return {
            'ids': torch.tensor(data["ids"], dtype=torch.long),
            'mask': torch.tensor(data["mask"], dtype=torch.long),
            'token_type_ids': torch.tensor(data["token_type_ids"], dtype=torch.long),
            'targets_start': torch.tensor(data["targets_start"], dtype=torch.long),
            'targets_end': torch.tensor(data["targets_end"], dtype=torch.long),
            'orig_tweet': data["orig_tweet"],
            'orig_selected': data["orig_selected"],
            'sentiment': data["sentiment"],
            'offsets': torch.tensor(data["offsets"], dtype=torch.long)
        }

In [8]:
def process_data2(tweet, selected_text, sentiment, tokenizer, max_len):
    tweet = " " + " ".join(str(tweet).split())
    selected_text = " " + " ".join(str(selected_text).split())

    len_st = len(selected_text) - 1
    idx0 = None
    idx1 = None

    for ind in (i for i, e in enumerate(tweet) if e == selected_text[1]):
        if " " + tweet[ind: ind+len_st] == selected_text:
            idx0 = ind
            idx1 = ind + len_st - 1
            break

    char_targets = [0] * len(tweet)
    if idx0 != None and idx1 != None:
        for ct in range(idx0, idx1 + 1):
            char_targets[ct] = 1
    
    tok_tweet = tokenizer.encode(tweet)
    input_ids_orig = tok_tweet.ids
    tweet_offsets = tok_tweet.offsets
    
    target_idx = []
    for j, (offset1, offset2) in enumerate(tweet_offsets):
        if sum(char_targets[offset1: offset2]) > 0:
            target_idx.append(j)
    
    targets_start = target_idx[0]
    targets_end = target_idx[-1]

    sentiment_id = {
        'positive': 1313,
        'negative': 2430,
        'neutral': 7974
    }
    
    input_ids = [0] + [sentiment_id[sentiment]] + [2] + [2] + input_ids_orig + [2]
    token_type_ids = [0, 0, 0, 0] + [0] * (len(input_ids_orig) + 1)
    mask = [1] * len(token_type_ids)
    tweet_offsets = [(0, 0)] * 4 + tweet_offsets + [(0, 0)]
    targets_start += 4
    targets_end += 4

    padding_length = max_len - len(input_ids)
    if padding_length > 0:
        input_ids = input_ids + ([1] * padding_length)
        mask = mask + ([0] * padding_length)
        token_type_ids = token_type_ids + ([0] * padding_length)
        tweet_offsets = tweet_offsets + ([(0, 0)] * padding_length)
    
    return {
        'ids': input_ids,
        'mask': mask,
        'token_type_ids': token_type_ids,
        'targets_start': targets_start,
        'targets_end': targets_end,
        'orig_tweet': tweet,
        'orig_selected': selected_text,
        'sentiment': sentiment,
        'offsets': tweet_offsets
    }


class TweetDataset2:
    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.tokenizer = TOKENIZER2
        self.max_len = MAX_LEN
    
    def __len__(self):
        return len(self.tweet)

    def __getitem__(self, item):
        data = process_data2(
            self.tweet[item], 
            self.selected_text[item], 
            self.sentiment[item],
            self.tokenizer,
            self.max_len
        )

        return {
            'ids': torch.tensor(data["ids"], dtype=torch.long),
            'mask': torch.tensor(data["mask"], dtype=torch.long),
            'token_type_ids': torch.tensor(data["token_type_ids"], dtype=torch.long),
            'targets_start': torch.tensor(data["targets_start"], dtype=torch.long),
            'targets_end': torch.tensor(data["targets_end"], dtype=torch.long),
            'orig_tweet': data["orig_tweet"],
            'orig_selected': data["orig_selected"],
            'sentiment': data["sentiment"],
            'offsets': torch.tensor(data["offsets"], dtype=torch.long)
        }

Added Sentiment Sampler

In [9]:
def jaccard(str1, str2):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a.intersection(b)
    return float(len(c)) / (len(a) + len(b) - len(c))

def calculate_jaccard_score(
    original_tweet, 
    target_string, 
    sentiment_val, 
    idx_start, 
    idx_end, 
    offsets
):
    if idx_end < idx_start:
        idx_end = idx_start
    
    filtered_output = ""
    for ix in range(idx_start, idx_end + 1):
        filtered_output += original_tweet[offsets[ix][0]: offsets[ix][1]]
        if ix+1 < len(offsets) and offsets[ix][1] < offsets[ix+1][0]:
            filtered_output += " "

    if sentiment_val == "neutral" or len(original_tweet.split()) < 2:
        filtered_output = original_tweet

    jac = jaccard(target_string.strip(), filtered_output.strip())
    return jac, filtered_output

In [10]:
df_test = pd.read_csv("/kaggle/input/competitions/tweet-sentiment-extraction/test.csv")
df_test["text"] = df_test["text"].fillna("").astype(str)
df_test["sentiment"] = df_test["sentiment"].fillna("neutral").astype(str)
df_test.loc[:, "selected_text"] = df_test["text"].values


training ig

In [11]:
device = torch.device("cuda")

model_config = AutoConfig.from_pretrained(MODEL_NAME)
model_config.output_hidden_states = True

model_config2 = transformers.RobertaConfig.from_pretrained(MODEL_NAME2)
model_config2.output_hidden_states = True

loss function: CE of start + end pos

5-fold training

In [12]:
model1 = TweetModel(conf=model_config)
model1.to(device)
model1.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-3/model_0.bin"))
model1.eval()

model2 = TweetModel(conf=model_config)
model2.to(device)
model2.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-3/model_1.bin"))
model2.eval()

model3 = TweetModel(conf=model_config)
model3.to(device)
model3.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-3/model_2.bin"))
model3.eval()

model4 = TweetModel(conf=model_config)
model4.to(device)
model4.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-3/model_3.bin"))
model4.eval()

model5 = TweetModel(conf=model_config)
model5.to(device)
model5.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-3/model_4.bin"))
model5.eval()

model6 = TweetModel2(conf=model_config2)
model6.to(device)
model6.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-2/model_0.bin"))
model6.eval()

model7 = TweetModel2(conf=model_config2)
model7.to(device)
model7.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-2/model_1.bin"))
model7.eval()

model8 = TweetModel2(conf=model_config2)
model8.to(device)
model8.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-2/model_2.bin"))
model8.eval()

model9 = TweetModel2(conf=model_config2)
model9.to(device)
model9.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-2/model_3.bin"))
model9.eval()

model0 = TweetModel2(conf=model_config2)
model0.to(device)
model0.load_state_dict(torch.load("/kaggle/input/datasets/cbdsigmas/tweet-model-2/model_4.bin"))
model0.eval()

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/datasets/abhishek/roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/datasets/abhishek/roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TweetModel2(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [13]:
def get_logits(models, loader):
    all_start, all_end = [], []
    all_offsets = []

    with torch.no_grad():
        for d in tqdm(loader):
            ids = d["ids"].to(device)
            token_type_ids = d["token_type_ids"].to(device)
            mask = d["mask"].to(device)

            outputs = [
                model(ids=ids, mask=mask, token_type_ids=token_type_ids)
                for model in models
            ]

            avg_start = sum(o[0] for o in outputs) / len(models)
            avg_end   = sum(o[1] for o in outputs) / len(models)

            all_start.append(avg_start.cpu())
            all_end.append(avg_end.cpu())
            all_offsets.append(d["offsets"].cpu())  # store offsets

    return (
        torch.cat(all_start),
        torch.cat(all_end),
        torch.cat(all_offsets)
    )

In [14]:
final_output = []

In [15]:
models_group1 = [model1, model2, model3, model4, model5]
models_group2 = [model6, model7, model8, model9, model0]

In [16]:
test_dataset = TweetDataset(
        tweet=df_test.text.values,
        sentiment=df_test.sentiment.values,
        selected_text=df_test.selected_text.values
    )

data_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=VALID_BATCH_SIZE,
    num_workers=1
)

test_dataset2 = TweetDataset2(
        tweet=df_test.text.values,
        sentiment=df_test.sentiment.values,
        selected_text=df_test.selected_text.values
    )

data_loader2 = torch.utils.data.DataLoader(
    test_dataset2,
    shuffle=False,
    batch_size=VALID_BATCH_SIZE,
    num_workers=1
)
all_start_1 = []
all_end_1 = []

with torch.no_grad():
    for d in tqdm(data_loader):
        ids = d["ids"].to(device)
        token_type_ids = d["token_type_ids"].to(device)
        mask = d["mask"].to(device)

        outputs = []
        for model in [model1, model2, model3, model4, model5]:
            start, end = model(ids=ids, mask=mask, token_type_ids=token_type_ids)
            outputs.append((start, end))

        avg_start = sum(o[0] for o in outputs) / 5
        avg_end   = sum(o[1] for o in outputs) / 5

        all_start_1.append(avg_start.cpu())
        all_end_1.append(avg_end.cpu())

all_start_2 = []
all_end_2 = []

with torch.no_grad():
    for d in tqdm(data_loader2):
        ids = d["ids"].to(device)
        token_type_ids = d["token_type_ids"].to(device)
        mask = d["mask"].to(device)

        outputs = []
        for model in [model6, model7, model8, model9, model0]:
            start, end = model(ids=ids, mask=mask, token_type_ids=token_type_ids)
            outputs.append((start, end))

        avg_start = sum(o[0] for o in outputs) / 5
        avg_end   = sum(o[1] for o in outputs) / 5

        all_start_2.append(avg_start.cpu())
        all_end_2.append(avg_end.cpu())

all_start_1 = torch.cat(all_start_1, dim=0)
all_end_1   = torch.cat(all_end_1, dim=0)

all_start_2 = torch.cat(all_start_2, dim=0)
all_end_2   = torch.cat(all_end_2, dim=0)

final_start = (all_start_1 + all_start_2) / 2
final_end   = (all_end_1 + all_end_2) / 2

final_start = torch.softmax(final_start, dim=1).numpy()
final_end   = torch.softmax(final_end, dim=1).numpy()

  0%|          | 0/442 [00:00<?, ?it/s]/tmp/ipykernel_24/613374741.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 442/442 [01:42<00:00,  4.33it/s]


In [17]:
start1, end1, offsets_all = get_logits(models_group1, data_loader)
start2, end2, _           = get_logits(models_group2, data_loader2)

final_start = (start1 + start2) / 2
final_end   = (end1 + end2) / 2

final_start = torch.softmax(final_start, dim=1).numpy()
final_end   = torch.softmax(final_end, dim=1).numpy()
offsets_all = offsets_all.numpy()

  0%|          | 0/442 [00:00<?, ?it/s]/tmp/ipykernel_24/613374741.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 442/442 [01:42<00:00,  4.33it/s]


In [18]:
final_output = []

for i in range(len(df_test)):
    tweet = df_test.text.values[i]
    selected_tweet = df_test.selected_text.values[i]
    sentiment = df_test.sentiment.values[i]

    idx_start = np.argmax(final_start[i])
    idx_end   = np.argmax(final_end[i])

    _, output_sentence = calculate_jaccard_score(
        original_tweet=tweet,
        target_string=selected_tweet,
        sentiment_val=sentiment,
        idx_start=idx_start,
        idx_end=idx_end,
        offsets=offsets_all[i]  # must come from correct tokenizer
    )

    output_sentence = pp(output_sentence, tweet)
    output_sentence = output_sentence.encode("ascii", "ignore").decode("ascii")

    final_output.append(output_sentence)

In [19]:
import re
import pandas as pd
import numpy as np

# Load submission template
df_submission = pd.read_csv("/kaggle/input/competitions/tweet-sentiment-extraction/sample_submission.csv")

processed_output = []

for i in range(len(final_output)):
    # 1. Force string type and strip existing edge whitespace
    pred_text = str(final_output[i]).strip()
    
    # 2. Check for letters (A-Z, a-z)
    if not re.search('[a-zA-Z]', pred_text):
        val = str(df_test.text.values[i]).strip()
    else:
        val = pred_text

    # 3. Handle NaNs or Empty strings resulting from ASCII removal
    if val == "" or val.lower() == "nan":
        val = str(df_test.text.values[i]).strip()
        if val.lower() == "nan": 
            val = "." # Absolute safety fallback

    # 4. Leading Space logic: Add " " before the target instance
    val = " " + val

    processed_output.append(val)

# 5. Save the final file
# Ensure length matches sample_submission to avoid Scoring Error
df_submission['selected_text'] = processed_output
df_submission.to_csv("submission.csv", index=False)

print(f"Submission saved with {len(df_submission)} rows.")

Submission saved with 3534 rows.
